# Download Environment Canada Weather Data

**Station:** Toronto Pearson International Airport (Station ID: 51459)  
**Period:** June 2013 - November 2025  
**Frequency:** Hourly  

**Purpose:** Download hourly weather data to merge with IESO electricity demand for ML modeling.

**Note:** Using Toronto weather as proxy for Ontario-wide weather patterns. Toronto region represents ~40% of Ontario's population and electricity demand.

In [1]:
# Import libraries for downloading weather data
import requests
import time
import os
import pandas as pd

print("Libraries loaded")

Libraries loaded


In [2]:
# Setup parameters for weather data download
station_id = 51459  # Toronto Pearson International Airport
weather_folder = "../../02_Datasets/raw/weather/"

# Create folder if it doesn't exist
os.makedirs(weather_folder, exist_ok=True)

# Build list of all months to download (June 2013 to November 2025)
download_list = []

# 2013: June to December only (weather data starts in June)
for month in range(6, 13):
    download_list.append((2013, month))

# 2014-2024: All 12 months
for year in range(2014, 2025):
    for month in range(1, 13):
        download_list.append((year, month))

# 2025: January to November (current month)
for month in range(1, 12):
    download_list.append((2025, month))

print(f"Station ID: {station_id}")
print(f"Download folder: {weather_folder}")
print(f"Total months to download: {len(download_list)}")
print(f"Date range: 2013-06 to 2025-11")

Station ID: 51459
Download folder: ../../02_Datasets/raw/weather/
Total months to download: 150
Date range: 2013-06 to 2025-11


In [3]:
# Download all weather files from Environment Canada
print("Starting weather data download...")
print("This will take about 3-4 minutes - downloading 150 files")
print("-" * 70)

successful = 0
failed = 0

for year, month in download_list:
    # Build download URL
    url = f"https://climate.weather.gc.ca/climate_data/bulk_data_e.html?format=csv&stationID={station_id}&Year={year}&Month={month}&timeframe=1&submit=Download+Data"
    
    # Set filename
    filename = f"weather_toronto_{year}_{month:02d}.csv"
    filepath = os.path.join(weather_folder, filename)
    
    try:
        # Download file
        response = requests.get(url, timeout=30)
        
        # Check if we got valid data (file should be at least 1KB)
        if response.status_code == 200 and len(response.content) > 1000:
            with open(filepath, 'wb') as f:
                f.write(response.content)
            print(f"✓ {year}-{month:02d}")
            successful += 1
        else:
            print(f"✗ {year}-{month:02d} - No data available")
            failed += 1
            
        # Wait 1 second between downloads to be respectful to server
        time.sleep(1)
        
    except Exception as e:
        print(f"✗ {year}-{month:02d} - Error: {e}")
        failed += 1

print("-" * 70)
print(f"\nDownload Summary:")
print(f"  Successful: {successful}")
print(f"  Failed: {failed}")
print(f"\n✓ Weather data saved to: {weather_folder}")

Starting weather data download...
This will take about 3-4 minutes - downloading 150 files
----------------------------------------------------------------------
✓ 2013-06
✓ 2013-07
✓ 2013-08
✓ 2013-09
✓ 2013-10
✓ 2013-11
✓ 2013-12
✓ 2014-01
✓ 2014-02
✓ 2014-03
✓ 2014-04
✓ 2014-05
✓ 2014-06
✓ 2014-07
✓ 2014-08
✓ 2014-09
✓ 2014-10
✓ 2014-11
✓ 2014-12
✓ 2015-01
✓ 2015-02
✓ 2015-03
✓ 2015-04
✓ 2015-05
✓ 2015-06
✓ 2015-07
✓ 2015-08
✓ 2015-09
✓ 2015-10
✓ 2015-11
✓ 2015-12
✓ 2016-01
✓ 2016-02
✓ 2016-03
✓ 2016-04
✓ 2016-05
✓ 2016-06
✓ 2016-07
✓ 2016-08
✓ 2016-09
✓ 2016-10
✓ 2016-11
✓ 2016-12
✓ 2017-01
✓ 2017-02
✓ 2017-03
✓ 2017-04
✓ 2017-05
✓ 2017-06
✓ 2017-07
✓ 2017-08
✓ 2017-09
✓ 2017-10
✓ 2017-11
✓ 2017-12
✓ 2018-01
✓ 2018-02
✓ 2018-03
✓ 2018-04
✓ 2018-05
✓ 2018-06
✓ 2018-07
✓ 2018-08
✓ 2018-09
✓ 2018-10
✓ 2018-11
✓ 2018-12
✓ 2019-01
✓ 2019-02
✓ 2019-03
✓ 2019-04
✓ 2019-05
✓ 2019-06
✓ 2019-07
✓ 2019-08
✓ 2019-09
✓ 2019-10
✓ 2019-11
✓ 2019-12
✓ 2020-01
✓ 2020-02
✓ 2020-03
✓ 2020-04
✓ 2020-0

In [4]:
# Check downloaded weather files
import glob

weather_files = sorted(glob.glob(weather_folder + "*.csv"))

print(f"Total weather files downloaded: {len(weather_files)}")
print(f"\nFirst 5 files:")
for f in weather_files[:5]:
    print(f"  {os.path.basename(f)}")
    
print(f"\nLast 5 files:")
for f in weather_files[-5:]:
    print(f"  {os.path.basename(f)}")

# Check one file to see what columns we have
print(f"\n" + "="*70)
print("Sample file structure (2024-01):")
sample_df = pd.read_csv(weather_files[-20])  # Load Jan 2024 as sample
print(f"\nColumns available: {len(sample_df.columns)}")
print(sample_df.columns.tolist())
print(f"\nFirst 3 rows:")
sample_df.head(3)

Total weather files downloaded: 150

First 5 files:
  weather_toronto_2013_06.csv
  weather_toronto_2013_07.csv
  weather_toronto_2013_08.csv
  weather_toronto_2013_09.csv
  weather_toronto_2013_10.csv

Last 5 files:
  weather_toronto_2025_07.csv
  weather_toronto_2025_08.csv
  weather_toronto_2025_09.csv
  weather_toronto_2025_10.csv
  weather_toronto_2025_11.csv

Sample file structure (2024-01):

Columns available: 31
['Longitude (x)', 'Latitude (y)', 'Station Name', 'Climate ID', 'Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)', 'Flag', 'Temp (°C)', 'Temp Flag', 'Dew Point Temp (°C)', 'Dew Point Temp Flag', 'Rel Hum (%)', 'Rel Hum Flag', 'Precip. Amount (mm)', 'Precip. Amount Flag', 'Wind Dir (10s deg)', 'Wind Dir Flag', 'Wind Spd (km/h)', 'Wind Spd Flag', 'Visibility (km)', 'Visibility Flag', 'Stn Press (kPa)', 'Stn Press Flag', 'Hmdx', 'Hmdx Flag', 'Wind Chill', 'Wind Chill Flag', 'Weather']

First 3 rows:


,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time (LST),Year,Month,Day,Time (LST),Flag,...,Wind Spd Flag,Visibility (km),Visibility Flag,Stn Press (kPa),Stn Press Flag,Hmdx,Hmdx Flag,Wind Chill,Wind Chill Flag,Weather
0,-79.63,43.68,TORONTO INTL A,6158731,2024-04-01 00:00,2024,4,1,00:00,NaN,...,NaN,24.1,NaN,99.38,NaN,NaN,NaN,NaN,NaN,NaN
1,-79.63,43.68,TORONTO INTL A,6158731,2024-04-01 01:00,2024,4,1,01:00,NaN,...,NaN,24.1,NaN,99.34,NaN,NaN,NaN,NaN,NaN,Mostly Cloudy
2,-79.63,43.68,TORONTO INTL A,6158731,2024-04-01 02:00,2024,4,1,02:00,NaN,...,NaN,24.1,NaN,99.26,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Load and merge all weather files
print("Loading all weather files...")
print("-" * 70)

all_weather = []

for i, file in enumerate(weather_files, 1):
    df = pd.read_csv(file)
    all_weather.append(df)
    
    if i % 20 == 0:  # Progress update every 20 files
        print(f"  Loaded {i}/{len(weather_files)} files...")

# Merge all months
weather_data = pd.concat(all_weather, ignore_index=True)

print("-" * 70)
print(f"✓ Loaded all {len(weather_files)} files")
print(f"\nTotal weather records: {len(weather_data):,}")
print(f"Date range: {weather_data['Date/Time (LST)'].min()} to {weather_data['Date/Time (LST)'].max()}")
print(f"Memory usage: {weather_data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Loading all weather files...
----------------------------------------------------------------------
  Loaded 20/150 files...
  Loaded 40/150 files...
  Loaded 60/150 files...
  Loaded 80/150 files...
  Loaded 100/150 files...
  Loaded 120/150 files...
  Loaded 140/150 files...
----------------------------------------------------------------------
✓ Loaded all 150 files

Total weather records: 109,584
Date range: 2013-06-01 00:00 to 2025-11-30 23:00
Memory usage: 63.7 MB


In [6]:
# Select only the columns we need for ML modeling
columns_to_keep = [
    'Date/Time (LST)',
    'Year', 
    'Month', 
    'Day',
    'Time (LST)',
    'Temp (°C)',
    'Dew Point Temp (°C)',
    'Rel Hum (%)',
    'Precip. Amount (mm)',
    'Wind Dir (10s deg)',
    'Wind Spd (km/h)',
    'Visibility (km)',
    'Stn Press (kPa)',
    'Hmdx',
    'Wind Chill'
]

weather_clean = weather_data[columns_to_keep].copy()

print("Selected weather features:")
print(weather_clean.columns.tolist())

print("\n" + "="*70)
print("Missing values per column:")
print(weather_clean.isnull().sum())

print("\n" + "="*70)
print("Missing values percentage:")
missing_pct = (weather_clean.isnull().sum() / len(weather_clean) * 100).round(2)
print(missing_pct[missing_pct > 0])

print("\n" + "="*70)
print("Basic statistics for temperature:")
print(weather_clean['Temp (°C)'].describe())

Selected weather features:
['Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Precip. Amount (mm)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'Hmdx', 'Wind Chill']

Missing values per column:
Date/Time (LST)             0
Year                        0
Month                       0
Day                         0
Time (LST)                  0
Temp (°C)                 768
Dew Point Temp (°C)       775
Rel Hum (%)               780
Precip. Amount (mm)    109584
Wind Dir (10s deg)       2677
Wind Spd (km/h)           770
Visibility (km)           766
Stn Press (kPa)           769
Hmdx                    90480
Wind Chill              86994
dtype: int64

Missing values percentage:
Temp (°C)                0.70
Dew Point Temp (°C)      0.71
Rel Hum (%)              0.71
Precip. Amount (mm)    100.00
Wind Dir (10s deg)       2.44
Wind Spd (km/h)          0.70
Visibility (km)          0.70
Stn Press (

In [7]:
# Drop columns with too much missing data
weather_clean = weather_clean.drop(columns=['Precip. Amount (mm)', 'Hmdx', 'Wind Chill'])

print("Dropped: Precipitation (100% missing), Humidex, Wind Chill")
print(f"Remaining columns: {weather_clean.columns.tolist()}")

# Create proper datetime column
weather_clean['DateTime'] = pd.to_datetime(weather_clean['Date/Time (LST)'])

# Sort by datetime to ensure chronological order
weather_clean = weather_clean.sort_values('DateTime').reset_index(drop=True)

# Forward fill small gaps (0.7% missing is likely just sensor issues for 1-2 hours)
weather_clean['Temp (°C)'] = weather_clean['Temp (°C)'].fillna(method='ffill')
weather_clean['Dew Point Temp (°C)'] = weather_clean['Dew Point Temp (°C)'].fillna(method='ffill')
weather_clean['Rel Hum (%)'] = weather_clean['Rel Hum (%)'].fillna(method='ffill')
weather_clean['Wind Spd (km/h)'] = weather_clean['Wind Spd (km/h)'].fillna(method='ffill')
weather_clean['Visibility (km)'] = weather_clean['Visibility (km)'].fillna(method='ffill')
weather_clean['Stn Press (kPa)'] = weather_clean['Stn Press (kPa)'].fillna(method='ffill')

# For wind direction, fill with 0 (calm/no wind)
weather_clean['Wind Dir (10s deg)'] = weather_clean['Wind Dir (10s deg)'].fillna(0)

print("\n" + "="*70)
print("After cleaning - Missing values:")
print(weather_clean.isnull().sum())

print("\n" + "="*70)
print(f"Clean weather dataset:")
print(f"  Records: {len(weather_clean):,}")
print(f"  Date range: {weather_clean['DateTime'].min()} to {weather_clean['DateTime'].max()}")
print(f"  Features: {len(weather_clean.columns)}")

Dropped: Precipitation (100% missing), Humidex, Wind Chill
Remaining columns: ['Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']

After cleaning - Missing values:
Date/Time (LST)          0
Year                     0
Month                    0
Day                      0
Time (LST)               0
Temp (°C)              249
Dew Point Temp (°C)    249
Rel Hum (%)            249
Wind Dir (10s deg)       0
Wind Spd (km/h)        249
Visibility (km)        249
Stn Press (kPa)        249
DateTime                 0
dtype: int64

Clean weather dataset:
  Records: 109,584
  Date range: 2013-06-01 00:00:00 to 2025-11-30 23:00:00
  Features: 13


C:\Users\nabee\AppData\Local\Temp\ipykernel_30364\2321926412.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  weather_clean['Temp (°C)'] = weather_clean['Temp (°C)'].fillna(method='ffill')
C:\Users\nabee\AppData\Local\Temp\ipykernel_30364\2321926412.py:15: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  weather_clean['Dew Point Temp (°C)'] = weather_clean['Dew Point Temp (°C)'].fillna(method='ffill')
C:\Users\nabee\AppData\Local\Temp\ipykernel_30364\2321926412.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  weather_clean['Rel Hum (%)'] = weather_clean['Rel Hum (%)'].fillna(method='ffill')
C:\Users\nabee\AppData\Local\Temp\ipykernel_30364\2321926412.py:17: FutureWarning: Series.fillna with 'method' is deprecated and w

In [9]:
# Use updated pandas syntax (ffill instead of fillna with method)
weather_clean['Temp (°C)'] = weather_clean['Temp (°C)'].ffill().bfill()
weather_clean['Dew Point Temp (°C)'] = weather_clean['Dew Point Temp (°C)'].ffill().bfill()
weather_clean['Rel Hum (%)'] = weather_clean['Rel Hum (%)'].ffill().bfill()
weather_clean['Wind Spd (km/h)'] = weather_clean['Wind Spd (km/h)'].ffill().bfill()
weather_clean['Visibility (km)'] = weather_clean['Visibility (km)'].ffill().bfill()  # Fixed!
weather_clean['Stn Press (kPa)'] = weather_clean['Stn Press (kPa)'].ffill().bfill()

print("Final check for missing values:")
print(weather_clean.isnull().sum().sum())

if weather_clean.isnull().sum().sum() == 0:
    print("✓ All missing values handled!")
else:
    print(f"⚠ Still {weather_clean.isnull().sum().sum()} missing values")
    print("\nDropping rows with any remaining missing values...")
    weather_clean = weather_clean.dropna()
    print(f"✓ Dropped {109584 - len(weather_clean)} rows")
    print(f"  Final records: {len(weather_clean):,}")

print("\n" + "="*70)
print("Weather data summary:")
print(f"  Total records: {len(weather_clean):,}")
print(f"  Date range: {weather_clean['DateTime'].min()} to {weather_clean['DateTime'].max()}")
print(f"  Features: {weather_clean.columns.tolist()}")

Final check for missing values:
0
✓ All missing values handled!

Weather data summary:
  Total records: 109,584
  Date range: 2013-06-01 00:00:00 to 2025-11-30 23:00:00
  Features: ['Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'DateTime']


In [10]:
# Keep only the columns we need (drop redundant ones)
final_weather = weather_clean[['DateTime', 'Temp (°C)', 'Dew Point Temp (°C)', 
                                 'Rel Hum (%)', 'Wind Dir (10s deg)', 
                                 'Wind Spd (km/h)', 'Visibility (km)', 
                                 'Stn Press (kPa)']].copy()

# Save to processed folder
output_path = "../../02_Datasets/processed/weather_toronto_2013_2025_cleaned.csv"
final_weather.to_csv(output_path, index=False)

print(f"✓ Saved cleaned weather data to:")
print(f"  {output_path}")
print(f"\nFile details:")
print(f"  Size: {os.path.getsize(output_path) / 1024**2:.1f} MB")
print(f"  Rows: {len(final_weather):,}")
print(f"  Columns: {final_weather.columns.tolist()}")
print(f"\nSample data:")
final_weather.head()

✓ Saved cleaned weather data to:
  ../../02_Datasets/processed/weather_toronto_2013_2025_cleaned.csv

File details:
  Size: 5.9 MB
  Rows: 109,584
  Columns: ['DateTime', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']

Sample data:


,DateTime,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Dir (10s deg),Wind Spd (km/h),Visibility (km),Stn Press (kPa)
0,2013-06-01 00:00:00,17.8,16.0,89.0,0.0,21.0,24.1,98.66
1,2013-06-01 01:00:00,17.8,16.0,89.0,0.0,21.0,24.1,98.66
2,2013-06-01 02:00:00,17.8,16.0,89.0,0.0,21.0,24.1,98.66
3,2013-06-01 03:00:00,17.8,16.0,89.0,0.0,21.0,24.1,98.66
4,2013-06-01 04:00:00,17.8,16.0,89.0,0.0,21.0,24.1,98.66
